# Production RAG Service — Notebook 1

## Ingestion Pipeline

Real application implementation:
PDF/DOCX extraction → Smart chunking → Metadata
→ multilingual E5 embeddings.

All code uses production application classes.

In [14]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent

assert (ROOT / "app").exists(), "Project root not found"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: D:\Agentic-Ai-Course\Porfolio1-production-rag-service


In [15]:
from uuid import uuid4
from io import BytesIO
from docx import Document

from app.ingestion.extractor import extract_document
from app.ingestion.chunker import (
    create_chunks, CHUNK_SIZE, CHUNK_OVERLAP
)
from app.ingestion.embeddings import EmbeddingService

print("Application imports PASSED")

Application imports PASSED


## 1. PDF Extraction

Uses the TechNova test PDF.
Extracted content retains page and optional section.

In [16]:
pdf = ROOT / "TechNova_HR_Policy_Manual.pdf"
assert pdf.exists(), f"Missing: {pdf}"

pdf_sections = extract_document(
    pdf.name, pdf.read_bytes()
)
assert pdf_sections

print("PDF sections:", len(pdf_sections))
for section in pdf_sections[:3]:
    print(
        "Page:", section.get("page"),
        "Section:", section.get("section")
    )
    print(section["text"][:200], "\n")

2026-11-25 12:02:11,299 | INFO     | app.ingestion.extractor | PDF extraction completed: 20 text/table sections
PDF sections: 20
Page: 1 Section: None
TechNova Solutions (Pvt.) Ltd. - HR Policy Manual v3.2 - Confidential
Page 1
 TechNova Solutions (Pvt.) Ltd.
 Employee HR Policy Manual
 Version 3.2 | Effective Date: 1 January 2026
 Owner: Human Reso 

Page: 2 Section: 1. Introduction and Employment Categories
TechNova Solutions (Pvt.) Ltd. - HR Policy Manual v3.2 - Confidential
Page 2
Table of Contents
1. Introduction and Employment Categories
2. Working Hours and Attendance
3. Leave Policy
4. Compensation 

Page: 3 Section: 1. Introduction and Employment Categories
TechNova Solutions (Pvt.) Ltd. - HR Policy Manual v3.2 - Confidential
Page 3
1. Introduction and Employment Categories
1.1 Purpose
This manual sets out the policies, rights and responsibilities that g 



## 2. DOCX Extraction

Creates an in-memory DOCX sample containing
paragraphs and a table.

In [17]:
doc = Document()
doc.add_heading("Example Company Policy", level=1)
doc.add_paragraph(
    "Employees receive 20 working days of annual leave."
)
table = doc.add_table(rows=2, cols=2)
table.cell(0, 0).text = "Grade"
table.cell(0, 1).text = "Allowance"
table.cell(1, 0).text = "G4"
table.cell(1, 1).text = "PKR 10000"

buffer = BytesIO()
doc.save(buffer)

docx_sections = extract_document(
    "sample.docx", buffer.getvalue()
)

assert docx_sections
print("DOCX extracted:", docx_sections)

DOCX extracted: [{'section': 'document', 'text': 'Example Company Policy\nEmployees receive 20 working days of annual leave.\nGrade | Allowance\nG4 | PKR 10000'}]


## 3. Smart Chunking

The application uses 800-character chunks with
150-character approximate overlap.
Metadata retains original document information.

In [18]:
chunks = create_chunks(
    document_id=uuid4(),
    filename=pdf.name,
    extracted_sections=pdf_sections
)
assert chunks

print("Chunk size:", CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)
print("Total chunks:", len(chunks))

for item in chunks[:3]:
    assert len(item["text"]) <= CHUNK_SIZE
    print(item["metadata"])
    print(item["text"][:150], "\n")

Chunk size: 800
Overlap: 150
Total chunks: 42
{'document_id': '41e1ca38-d3e7-48ef-8c8d-900f804901b4', 'filename': 'TechNova_HR_Policy_Manual.pdf', 'page': 1, 'section': None, 'chunk_position': 0, 'section_chunk_position': 0}
TechNova Solutions (Pvt.) Ltd. - HR Policy Manual v3.2 - Confidential Page 1 TechNova Solutions (Pvt.) Ltd. Employee HR Policy Manual Version 3.2 | Ef 

{'document_id': '41e1ca38-d3e7-48ef-8c8d-900f804901b4', 'filename': 'TechNova_HR_Policy_Manual.pdf', 'page': 2, 'section': '1. Introduction and Employment Categories', 'chunk_position': 1, 'section_chunk_position': 0}
TechNova Solutions (Pvt.) Ltd. - HR Policy Manual v3.2 - Confidential Page 2 Table of Contents 1. Introduction and Employment Categories 2. Working Ho 

{'document_id': '41e1ca38-d3e7-48ef-8c8d-900f804901b4', 'filename': 'TechNova_HR_Policy_Manual.pdf', 'page': 3, 'section': '1. Introduction and Employment Categories', 'chunk_position': 2, 'section_chunk_position': 0}
TechNova Solutions (Pvt.) Ltd. - 

## 4. Embeddings

Uses the application's real multilingual E5 model.
Initial download may require internet access.

In [19]:
samples = [item["text"] for item in chunks[:3]]
vectors = EmbeddingService.embed_documents(samples)

assert len(vectors) == len(samples)
assert all(len(vector) == 384 for vector in vectors)

query_vector = EmbeddingService.embed_query(
    "How many annual leave days?"
)
assert len(query_vector) == 384

print("Document vectors:", len(vectors))
print("Vector dimensions:", len(vectors[0]))
print("Query dimensions:", len(query_vector))
print("NOTEBOOK 1: PASSED")

2026-11-25 12:02:11,416 | INFO     | app.ingestion.embeddings | Generating embeddings for 3 chunks
2026-11-25 12:02:11,804 | INFO     | app.ingestion.embeddings | Successfully embedded 3 chunks
2026-11-25 12:02:11,804 | INFO     | app.ingestion.embeddings | Generating query embedding
2026-11-25 12:02:11,876 | INFO     | app.ingestion.embeddings | Query embedding generated successfully
Document vectors: 3
Vector dimensions: 384
Query dimensions: 384
NOTEBOOK 1: PASSED
